# 2026/2027 年末年始 日当直調整

12/28 当直 〜 1/3 当直の 13 枠を、12/29・12/30・1/3 を日当直(24h)にまとめて **10 枠**にする。

- 指定医（オンコール 3 名 + その他の指定医 7 名）= 10 名 → **各枠にちょうど 1 名**
- 非指定医 5 名は必ず指定医とペアにする（非指定医が院内、指定医がオンコール）
  - オンコール専任 3 名には必ず非指定医を付ける → 3 枠
  - 残り 2 名の非指定医は指定医 2 名と組む（その指定医はオンコール側）→ 2 枠
  - 残る指定医 5 名は単独で院内 → 5 枠
- 全員ちょうど 1 枠、フォームの希望順位（二次変換）の合計を最小化（オンコール 3 名分は `w_oc` 倍に重み付け）

In [7]:
import pandas as pd, numpy as np
from pulp import LpProblem, LpVariable, LpMinimize, LpStatus, lpSum, value
from script.drive_io import load_config, get_services, SCOPE_ALL
from script.helper import read_form_response, make_solver, warn_if_not_proven_optimal

In [8]:
# Parameters
year_plan, month_plan = 2027, 'ny'
l_member_desig = ['藤川 慎也','熊倉 陽介','越山 太輔','森田 進','庄司 瑛武','清田 正紘','宇野 晃人']
l_member_nondesig = ['藤由 彩香','水上 颯','山内 正隆','緒方 優','秦 美能理']
l_member_oc = ['岡田 直大', '榊原 英輔', '市橋 香代']
l_duty = ['12/28 当直','12/29 日当直','12/30 日当直','12/31 日直','12/31 当直','1/1 日直','1/1 当直','1/2 日直','1/2 当直','1/3 日当直']

# Weight on the on-call doctors' preference cost relative to everyone else's (1 = equal).
# Checked on this year's responses (on-call ranks: 岡田/榊原/市橋):
#   1      -> 3/1/5, others' worst = 4th
#   2 - 5  -> 3/1/1, others' worst = 5th
#   10+    -> 1/1/2, others' worst = 5th (three others at 4th)
w_oc = 2

# Hard exclusions (member, duty): never assign. Taken from the free-text comments.
l_ng = [
    ('秦 美能理', '12/31 日直'),   # 「12/31は予定あり勤務が難しい」
    ('秦 美能理', '12/31 当直'),
]

col_name = 'お名前（敬称略）'
col_comment = 'その他、ご意見等'
prefix_pref = '日当直希望'

l_member = l_member_desig + l_member_nondesig + l_member_oc
assert len(set(l_member)) == len(l_member), 'duplicated member name'
# Every slot needs exactly one designated doctor (on-site alone, or on-call over a non-designated
# doctor), and every doctor works exactly once -> the two counts must match.
assert len(l_member_desig) + len(l_member_oc) == len(l_duty), \
    f'designated ({len(l_member_desig)}) + on-call ({len(l_member_oc)}) must equal number of slots ({len(l_duty)})'
assert len(l_member_nondesig) <= len(l_member_desig) + len(l_member_oc)
assert len(l_member_nondesig) >= len(l_member_oc), 'every on-call doctor needs a non-designated doctor'

In [9]:
# Read Google Form responses
config = load_config()
services = get_services(config, SCOPE_ALL)
path_form = f'/dutyshift/result/{year_plan}/{month_plan}/form_{year_plan}{month_plan}'
d_preference_src = read_form_response(services, path_form)
# Keep each doctor's last submission only
d_preference_src = d_preference_src.sort_values(by = 'Timestamp').drop_duplicates(subset = [col_name], keep = 'last').reset_index(drop = True)

l_answered = d_preference_src[col_name].tolist()
print(f'Responses: {len(l_answered)}')
print('Not answered (treated as no preference):', [m for m in l_member if m not in l_answered])
print('Answered but not in any member list (ignored):', [m for m in l_answered if m not in l_member])
print()
for _, row in d_preference_src.iterrows():
    if isinstance(row[col_comment], str) and row[col_comment].strip():
        print(f'{row[col_name]}: {row[col_comment]}')

Responses: 15
Not answered (treated as no preference): []
Answered but not in any member list (ignored): []

市橋 香代: 難しい調整お疲れさまです。
秦 美能理: 12/31に第9,10希望をつけていますが、予定あり勤務が難しいです。
岡田 直大: 誠に申し訳ございません。もし可能そうでしたら、第1-4希望の中でご調整いただけましたら幸いです。


In [10]:
# Convert '第N希望' -> rank N-1 (0 = first choice). index: duty, columns: member.
def convert_preference(d_preference_src, l_member, l_duty):
    d_rank = pd.DataFrame(np.nan, index = l_duty, columns = l_member)
    for member in l_member:
        rows = d_preference_src.loc[d_preference_src[col_name] == member]
        if len(rows) == 0:
            continue
        row = rows.iloc[-1]
        for duty in l_duty:
            col = f'{prefix_pref}[{duty}]'
            if col not in row.index:
                raise KeyError(f'Form has no question for {duty!r} (expected column {col!r})')
            ans = row[col]
            if isinstance(ans, str) and ans.startswith('第') and ans.endswith('希望'):
                d_rank.loc[duty, member] = int(ans[1:-2]) - 1
    return d_rank

d_rank = convert_preference(d_preference_src, l_member, l_duty)
# Missing answers (non-respondent or blank) -> no preference
d_rank_filled = d_rank.fillna(0)

# Quadratic penalty: rank r (0-based) -> r(r+1)/2, i.e. 1st=0, 2nd=1, 3rd=3, 4th=6, ...
d_cost = d_rank_filled * (d_rank_filled + 1) / 2
d_cost[l_member_oc] *= w_oc

# 1-based rank table for review
display((d_rank + 1).astype('Int64').T)

,12/28 当直,12/29 日当直,12/30 日当直,12/31 日直,12/31 当直,1/1 日直,1/1 当直,1/2 日直,1/2 当直,1/3 日当直
藤川 慎也,5,7,6,4,9,8,3,2,1,10
熊倉 陽介,5,2,3,7,8,10,9,6,4,1
越山 太輔,1,3,5,4,2,9,7,8,6,10
森田 進,1,3,5,7,9,10,8,6,4,2
庄司 瑛武,8,10,9,5,6,4,3,1,2,7
清田 正紘,9,10,1,5,6,8,7,4,3,2
宇野 晃人,2,3,6,8,10,9,7,5,4,1
藤由 彩香,1,4,5,6,7,10,9,8,2,3
水上 颯,9,10,1,2,3,6,5,4,7,8
山内 正隆,9,10,8,2,6,3,5,1,4,7


In [11]:
# Optimize assignment
prob = LpProblem('ny_assign', LpMinimize)
x = {(d, m): LpVariable(f'x_{i_d}_{i_m}', cat = 'Binary') for i_d, d in enumerate(l_duty) for i_m, m in enumerate(l_member)}

# Objective: minimize total preference cost
prob += lpSum(d_cost.loc[d, m] * x[d, m] for d in l_duty for m in l_member)

# Each member works exactly one slot
for m in l_member:
    prob += lpSum(x[d, m] for d in l_duty) == 1

for d in l_duty:
    n_desig = lpSum(x[d, m] for m in l_member_desig)
    n_nondesig = lpSum(x[d, m] for m in l_member_nondesig)
    n_oc = lpSum(x[d, m] for m in l_member_oc)
    # Exactly one designated doctor (designated or on-call) per slot
    prob += n_desig + n_oc == 1
    # At most one non-designated doctor per slot
    prob += n_nondesig <= 1
    # On-call doctors are never on-site, so they always need a non-designated doctor on-site
    prob += n_oc <= n_nondesig

# Hard exclusions
for m, d in l_ng:
    prob += x[d, m] == 0

prob.solve(make_solver(msg = False))
print('Status:', LpStatus[prob.status], '/ objective:', value(prob.objective))
warn_if_not_proven_optimal(prob, 'ny_assign')
assert LpStatus[prob.status] == 'Optimal', 'No feasible assignment -- check l_ng and member lists'

Status: Optimal / objective: 31.0


In [12]:
# Output assignment result
def label(m):
    return f'{m}（第{int(d_rank.loc[d, m]) + 1}希望）' if not pd.isna(d_rank.loc[d, m]) else f'{m}（希望なし）'

l_row = []
for d in l_duty:
    l_assigned = [m for m in l_member if x[d, m].varValue > 0.5]
    l_nd = [m for m in l_assigned if m in l_member_nondesig]
    l_ds = [m for m in l_assigned if m not in l_member_nondesig]
    if l_nd:   # non-designated on-site, designated / on-call doctor on call
        l_row.append({'日当直': label(l_nd[0]), 'オンコール': label(l_ds[0])})
    else:      # designated doctor on-site alone
        l_row.append({'日当直': label(l_ds[0]), 'オンコール': ''})
duty_assignment = pd.DataFrame(l_row, index = l_duty)
display(duty_assignment)

# Distribution of obtained preference ranks
s_rank_got = pd.Series({m: d_rank.loc[d, m] + 1 for d in l_duty for m in l_member if x[d, m].varValue > 0.5})
print(s_rank_got.value_counts(dropna = False).sort_index().rename('人数').rename_axis('第N希望'))

,日当直,オンコール
12/28 当直,藤由 彩香（第1希望）,市橋 香代（第1希望）
12/29 日当直,熊倉 陽介（第2希望）,
12/30 日当直,水上 颯（第1希望）,清田 正紘（第1希望）
12/31 日直,緒方 優（第1希望）,榊原 英輔（第1希望）
12/31 当直,越山 太輔（第2希望）,
1/1 日直,庄司 瑛武（第4希望）,
1/1 当直,藤川 慎也（第3希望）,
1/2 日直,山内 正隆（第1希望）,宇野 晃人（第5希望）
1/2 当直,秦 美能理（第3希望）,岡田 直大（第3希望）
1/3 日当直,森田 進（第2希望）,


第N希望
1.0    7
2.0    3
3.0    3
4.0    1
5.0    1
Name: 人数, dtype: int64
